# 环境和分布偏移（Environment and Distribution Shift）

## 1. 什么是分布偏移？

**分布偏移（Distribution Shift）** 是指训练数据与实际应用数据的分布不同，导致模型的预测性能可能下降。

例如：一个模型在训练集和测试集上表现良好，但实际部署后，由于环境或数据发生变化，预测准确率可能大幅降低。

因此，**训练集准确率高，并不意味着模型在真实环境中一定表现良好。**

## 2. 分布偏移的类型

主要包括三种：**协变量偏移、标签偏移和概念偏移**。

### 2.1 协变量偏移（Covariate Shift）

**输入特征的分布发生变化，但输入与标签之间的关系保持不变。**

数学表示：

$$
P_S(X)\neq P_T(X)
$$

$$
P_S(Y\mid X)=P_T(Y\mid X)
$$

其中：

- $P_S$：训练数据分布。
- $P_T$：目标数据分布。
- $X$：输入特征。
- $Y$：标签。

**示例：猫狗图像分类**

训练数据：真实猫狗照片。

![猫狗训练集](../img/cat-dog-train.svg)

测试数据：卡通猫狗图片。

![猫狗测试集](../img/cat-dog-test.svg)

两种数据的图像风格不同，但猫和狗的类别定义没有改变。

因此，模型可能因输入特征分布变化而导致准确率下降。

### 2.2 标签偏移（Label Shift）

**标签的分布发生变化，但每个类别对应的特征分布保持不变。**

$$
P_S(Y)\neq P_T(Y)
$$

$$
P_S(X\mid Y)=P_T(X\mid Y)
$$

例如：

- 训练集：猫占 50%，狗占 50%。
- 测试集：猫占 80%，狗占 20%。

虽然类别比例发生变化，但假设每个类别内部的图像特征分布保持不变。

### 2.3 概念偏移（Concept Shift）

**输入特征与标签之间的关系发生变化。**

$$
P_S(Y\mid X)\neq P_T(Y\mid X)
$$

例如，美国不同地区对软饮料的称呼可能不同。

<img src="../img/popvssoda.png" width="400">

同一种饮料，在不同地区可能被称为不同的名称，导致相同输入对应的标签发生变化。

概念偏移还可能发生在垃圾邮件识别、疾病诊断标准和用户行为预测等任务中。

### 2.4 三种分布偏移的区别

| 类型 | 变化内容 | 不变内容 |
|---|---|---|
| 协变量偏移 | $P(X)$ | $P(Y\mid X)$ |
| 标签偏移 | $P(Y)$ | $P(X\mid Y)$ |
| 概念偏移 | $P(Y\mid X)$ | 无统一要求 |

## 3. 分布偏移的实际案例

### 3.1 医学诊断

假设使用年轻健康人的数据和老年患者的数据训练疾病检测模型。

模型可能学到的是**年龄差异，而不是疾病特征**。

因此，即使训练准确率很高，也不代表能够准确诊断疾病。

### 3.2 自动驾驶

使用游戏引擎生成的道路图像训练模型，在真实道路上可能表现较差。

原因是虚拟环境与真实环境的纹理、光照等特征存在差异。

### 3.3 非平稳分布（Nonstationary Distribution）

**数据分布随时间不断变化。**

例如：

- 垃圾邮件的内容和形式不断变化。
- 用户的购物偏好随季节变化。
- 新产品出现后，原有推荐模型可能失效。

因此，模型可能需要定期使用新数据进行更新。

## 4. 经验风险与真实风险

### 4.1 经验风险（Empirical Risk）

经验风险是模型在**已有训练数据上的平均损失**。

$$
R_{\mathrm{emp}}=\frac{1}{n}\sum_{i=1}^{n}l(f(x_i),y_i)
$$

其中：

- $n$：训练样本数量。
- $f(x_i)$：模型预测值。
- $y_i$：真实标签。
- $l$：损失函数。

模型训练通常通过最小化经验风险来优化参数。

### 4.2 真实风险（True Risk）

真实风险是模型在**真实数据分布上的期望损失**。

$$
R_{\mathrm{true}}=\mathbb{E}_{(X,Y)\sim P}[l(f(X),Y)]
$$

由于通常无法获得真实环境中的全部数据，因此使用经验风险近似真实风险。

**核心区别：**

- 经验风险：训练数据上的平均损失。
- 真实风险：真实数据分布上的期望损失。

如果训练数据与实际数据分布不同，经验风险较低也不代表真实风险较低。

## 5. 分布偏移纠正

### 5.1 协变量偏移纠正

当训练数据与目标数据的特征分布不同，但 $P(Y\mid X)$ 不变时，可以通过**重要性加权（Importance Weighting）**进行纠正。

为每个训练样本计算权重：

$$
\beta_i=\frac{p(x_i)}{q(x_i)}
$$

其中：

- $p(x)$：目标数据的特征分布。
- $q(x)$：训练数据的特征分布。
- $\beta_i$：样本的重要性权重。

加权后的训练目标：

$$
\min_f\frac{1}{n}\sum_{i=1}^{n}\beta_i l(f(x_i),y_i)
$$

**核心思想：提高目标分布中常见样本的训练权重，降低相对不常见样本的权重。**

实际应用中，可以训练一个二分类器来区分训练数据和目标数据，再估计重要性权重。

需要注意，该方法要求目标分布覆盖的区域在训练分布中也有一定概率出现。

### 5.2 标签偏移纠正

当类别比例发生变化，但 $P(X\mid Y)$ 保持不变时，可以根据类别分布调整样本权重。

$$
\beta_i=\frac{p(y_i)}{q(y_i)}
$$

例如，测试环境中某一类别出现得更加频繁，可以适当提高该类别训练样本的权重。

目标数据的标签分布可以利用已有分类器、验证集混淆矩阵和目标数据的预测结果进行估计。

### 5.3 概念偏移纠正

当输入与标签之间的关系发生变化时，通常需要：

1. 收集新的标注数据。
2. 使用新数据更新模型参数。
3. 必要时重新训练模型。

概念偏移通常比前两种偏移更难处理。

## 6. 机器学习问题的分类

### 6.1 批量学习（Batch Learning）

使用固定的数据集训练模型，然后部署到实际环境中。

特点：

- 训练数据通常提前收集完成。
- 模型部署后不会自动持续更新。
- 适合数据分布相对稳定的任务。

### 6.2 在线学习（Online Learning）

模型持续接收新数据，并根据新的观测结果更新参数。

基本流程：

**接收数据 → 模型预测 → 获得真实结果 → 计算损失 → 更新模型**

适用于数据不断变化的任务，例如股票预测、广告推荐等。

### 6.3 多臂老虎机（Multi-Armed Bandits）

模型在有限的行动中进行选择，通过不断探索和利用已有信息来获得更高的奖励。

例如：选择哪种广告能够获得更高的点击率。

### 6.4 控制（Control）

模型根据环境状态以及过去的行动调整决策。

例如：通过控制器自动调节设备温度。

### 6.5 强化学习（Reinforcement Learning）

模型通过与环境交互，根据获得的奖励学习决策策略，以最大化长期收益。

例如：机器人控制、游戏 AI 和自动驾驶决策。

## 7. 机器学习中的公平、责任和透明度

机器学习模型不仅用于预测，还可能影响现实世界中的决策。

因此，模型部署需要考虑：

- **公平性**：避免对不同群体产生不合理的差别影响。
- **可靠性**：确保模型在实际环境中表现稳定。
- **透明度**：理解模型预测和决策的依据。
- **反馈循环**：防止模型的决策影响未来数据，进而强化原有偏差。

例如，推荐系统会影响用户看到的内容，而用户的行为又会成为下一轮训练数据。

因此，需要关注模型与环境之间的相互影响。

## 8. 总结

1. **分布偏移**：训练数据与实际数据分布不同，可能导致模型性能下降。
2. **协变量偏移**：输入特征分布变化，标签条件分布不变。
3. **标签偏移**：类别比例变化，类别条件特征分布不变。
4. **概念偏移**：输入与标签之间的关系发生变化。
5. **经验风险**：训练数据上的平均损失。
6. **真实风险**：真实数据分布上的期望损失。
7. **纠正方法**：重要性加权、调整类别权重、更新或重新训练模型。
8. **模型部署**：需要持续监控数据变化，并考虑公平性和反馈循环。

**核心思想：机器学习模型不仅要在训练数据上表现良好，还需要适应真实环境中的数据变化，才能具备可靠的泛化能力。**